[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/vectors/t03_lines.ipynb)

# Lines in 3D: meet or miss?

An exam question asks where the lines r = (1, −1, 2) + λ(2, 3, 1) and r = (2, 1, 0) + μ(1, 2, −1) meet. Match x and
y, and you get λ = 0, μ = −1 and the point (1, −1, 2). But the second line gives (1, −1, 1) there: the heights differ,
the lines never meet, and they miss by 1/√35 ≈ 0.17.

In this notebook you can redo every problem from the video, and check any pair of lines of your own: parallel, meet
or skew, with the angle and the shortest distance.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When a slider appears, drag it and watch. Change any
number and run the cell again: nothing can break. It works in your browser (no install) and in Google Colab.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look: one style for every figure in this series of notebooks. Run this cell first.
import math
from fractions import Fraction
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from ipywidgets import interact, FloatSlider, Text, Button, Output, HBox, VBox, Layout

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM = "#ece6d6", "#93a097"           # chalk, and chalk smudge
CORAL = "#e3896b"                           # the one key thing in each figure
TEAL, SAGE, LILAC = "#6fb6bb", "#a0c27c", "#c9a0d3"   # along (dot) · the answer · across (cross, the shortest link)

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#3b4c47", "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "legend.facecolor": SLATE2, "legend.edgecolor": DIM, "legend.labelcolor": CHALK, "legend.framealpha": 0.9,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

def dot(a, b): return sum(x * y for x, y in zip(a, b))
def cross(a, b): return (a[1] * b[2] - a[2] * b[1], a[2] * b[0] - a[0] * b[2], a[0] * b[1] - a[1] * b[0])
def length(a): return math.sqrt(dot(a, a))
def sub(a, b): return tuple(x - y for x, y in zip(a, b))
def at(a, b, t): return tuple(x + t * y for x, y in zip(a, b))

def nice(v):
    """A number as the board shows it: 3, −1, 0.17, with simple fractions kept exact (2/3)."""
    f = Fraction(v).limit_denominator(12)
    if abs(float(f) - v) < 1e-9: s = str(f)
    else: s = f"{v:.2f}".rstrip("0").rstrip(".")
    return s.replace("-", "−")

def show(v): return "(" + ", ".join(nice(x) for x in v) + ")"

def column(fig, x, y, vals, color=CHALK, size=15, label=""):
    """An arrow's column of numbers, in a bracket box, at figure position (x, y)."""
    fig.text(x, y, "\n".join(nice(v) for v in vals), color=color, fontsize=size, va="center", ha="center",
             family="serif", linespacing=1.45, bbox=dict(boxstyle="square,pad=0.4", fc=SLATE, ec=DIM, lw=1.2))
    if label: fig.text(x, y + 0.13, label, color=DIM, fontsize=12, ha="center", family="serif")
    fig.text(x + 0.06, y, ".", color=SLATE, fontsize=size)          # keeps the box's edge inside a cropped figure

def space(ax, elev, azim):
    ax.set_facecolor(SLATE)
    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis.set_pane_color((0.16, 0.22, 0.2, 0.6)); axis.label.set_color(DIM)
    ax.view_init(elev=elev, azim=azim)
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")

def closest(a1, b1, a2, b2):
    """The closest points of two lines that aren't parallel (the link between them is square to both)."""
    r = sub(a2, a1)
    A, B, C = dot(b1, b1), dot(b1, b2), dot(b2, b2)
    det = A * C - B * B
    lam = (dot(r, b1) * C - dot(r, b2) * B) / det
    mu = (dot(r, b1) * B - dot(r, b2) * A) / det
    return at(a1, b1, lam), at(a2, b2, mu)

## 1 · The trap: match x and y, then test z

Two lines in space give three equations (x, y and z) but only two unknowns (λ and μ). Two equations in two unknowns
nearly always have an answer, so the third equation is the test.

In [ ]:
a1, b1, a2, b2 = (1, -1, 2), (2, 3, 1), (2, 1, 0), (1, 2, -1)
lam, mu = 0, -1                     # x: 1 + 2λ = 2 + μ,  y: −1 + 3λ = 1 + 2μ
print("line 1 at λ = 0: ", show(at(a1, b1, lam)))
print("line 2 at μ = −1:", show(at(a2, b2, mu)))
print("x and y agree; z is", at(a1, b1, lam)[2], "against", at(a2, b2, mu)[2], "→ the lines never meet")

## 2 · The method: which tool?

1. **Write every line as a point and a direction.**
2. **Ask which tool the question needs.** An angle: a dot product of the directions. A gap: a cross product. A
   meeting point: all three equations.
3. **A point meeting a line at a right angle**: make a dot product zero.
4. **Check with a number.**

## 3 · The line through two points

A = (1, −2, 3) is fixed. Drag B, and watch the arrow B − A and its column of numbers: the direction of the line. The
direction cosines are the column divided by its length. The view sliders turn the picture.

In [ ]:
def two_points(bx=3.0, by=-1.0, bz=1.0, turn=-60, tilt=20):
    A, B = (1, -2, 3), (bx, by, bz)
    d = sub(B, A)
    L = length(d)
    fig = plt.figure(figsize=(9, 5.6))
    ax = fig.add_axes([0.0, 0.0, 0.68, 1.0], projection="3d")
    space(ax, tilt, turn)
    if L > 0:
        ts = np.array([-1.5, 2.5])
        ax.plot(A[0] + ts * d[0], A[1] + ts * d[1], A[2] + ts * d[2], color=DIM, lw=1.4, ls="--")
        ax.quiver(*A, *d, color=CHALK, lw=2.4, arrow_length_ratio=0.15)
    ax.scatter(*A, color=CHALK, s=40); ax.scatter(*B, color=CORAL, s=60, depthshade=False)
    ax.text(*A, "  A", color=CHALK); ax.text(*B, "  B", color=CORAL)
    ax.set_xlim(-4, 6); ax.set_ylim(-6, 4); ax.set_zlim(-3, 7)
    column(fig, 0.74, 0.55, d, label="B − A")
    if L > 0:
        column(fig, 0.86, 0.55, [v / L for v in d], color=CORAL, label="cosines")
        ax.set_title(f"direction {show(d)}, length {nice(L)}", color=CHALK)
    plt.show()

interact(two_points, bx=FloatSlider(value=3, min=-3, max=5, step=1, description="B: x"),
         by=FloatSlider(value=-1, min=-5, max=3, step=1, description="B: y"),
         bz=FloatSlider(value=1, min=-2, max=6, step=1, description="B: z"),
         turn=FloatSlider(value=-60, min=-180, max=180, step=10, description="view: turn"),
         tilt=FloatSlider(value=20, min=-10, max=80, step=5, description="view: tilt"));

## 4 · Angles: only the directions matter

cos θ = |b₁ · b₂| ÷ (|b₁||b₂|). A line runs both ways, so the formula keeps the magnitude. And two lines are
perpendicular when the dot product of their directions is zero.

In [ ]:
g1, g2 = (1, 2, 2), (-2, 1, -2)
print(f"dot = {dot(g1, g2)}, so cos θ = {abs(dot(g1, g2))}/{length(g1):g}×{length(g2):g} = {nice(abs(dot(g1, g2)) / 9)}"
      f", θ ≈ {math.degrees(math.acos(4 / 9)):.1f}°")
for p in range(5, 10):
    print(f"p = {p}: (1, p, 2) · (3, −1, 2) = {dot((1, p, 2), (3, -1, 2))}", "← perpendicular" if dot((1, p, 2), (3, -1, 2)) == 0 else "")

## 5 · Meet, skew or parallel, and the shortest distance

Compare the directions first. Not multiples: match x and y, then test z. Skew: d = |(a₂ − a₁) · (b₁ × b₂)| ÷
|b₁ × b₂|. Parallel: d = |b × (a₂ − a₁)| ÷ |b|.

In [ ]:
print("meet:  (1, 0, −1) + 2(1, 1, 2) =", show(at((1, 0, -1), (1, 1, 2), 2)),
      "  and (1, 3, 2) + 1(2, −1, 1) =", show(at((1, 3, 2), (2, -1, 1), 1)))
n = cross(b1, b2); w = sub(a2, a1)
print(f"trap pair: b₁ × b₂ = {show(n)}, (a₂ − a₁) · (b₁ × b₂) = {dot(w, n)}, d = 1/√{dot(n, n)} = {abs(dot(w, n)) / length(n):.3f}")
q = cross((2, -1, 2), sub((3, 1, 2), (1, 2, -1)))
print(f"parallel: b × (a₂ − a₁) = {show(q)}, d = √{dot(q, q)} ÷ 3 = {length(q) / 3:.3f}")

## 6 · The foot of the perpendicular, and the image of a point

From P = (6, 4, 0) to the line (x − 1)/2 = y/1 = (z − 2)/(−2). A general point is (1 + 2λ, λ, 2 − 2λ); the arrow from P
to it is square to the line when its dot product with (2, 1, −2) is zero. The foot is the midpoint of P and its image.

In [ ]:
P, A, b = (6, 4, 0), (1, 0, 2), (2, 1, -2)
for t in range(0, 4):
    print(f"λ = {t}: (Q − P) · b = {dot(sub(at(A, b, t), P), b)}")
Q = at(A, b, 2)
print("foot:", show(Q), "  distance:", nice(length(sub(Q, P))), "  image = 2Q − P =", show(sub(tuple(2 * v for v in Q), P)))

## 7 · Check your answer

Type each line as its point and its direction: `1 -1 2 ; 2 3 1` means r = (1, −1, 2) + λ(2, 3, 1). If you have a
meeting point, type it too. Press **check**: the cell says whether the lines are parallel, meet or are skew, with the
angle and the shortest distance, and whether your point is on both lines.

In [ ]:
def numbers(s):
    s = s.replace("−", "-").replace(",", " ").replace("(", " ").replace(")", " ").replace(";", " ")
    return [float(v) for v in s.split()]

def classify(a1, b1, a2, b2):
    n, w = cross(b1, b2), sub(a2, a1)
    cos = abs(dot(b1, b2)) / (length(b1) * length(b2))
    angle = math.degrees(math.acos(min(1.0, cos)))
    if length(n) < 1e-12:
        d = length(cross(b1, w)) / length(b1)
        return ("the same line" if d < 1e-9 else "parallel"), angle, d
    d = abs(dot(w, n)) / length(n)
    return ("they meet" if d < 1e-9 else "skew"), angle, d

def on_line(p, a, b):
    return length(cross(b, sub(p, a))) < 1e-9

def check_your_answer(line_1, line_2, point=""):
    try:
        p, q = numbers(line_1), numbers(line_2)
        a1, b1, a2, b2 = tuple(p[:3]), tuple(p[3:6]), tuple(q[:3]), tuple(q[3:6])
        assert len(p) == 6 and len(q) == 6 and length(b1) > 0 and length(b2) > 0
    except Exception:
        print("Type six numbers for each line: the point, then the direction."); return
    kind, angle, d = classify(a1, b1, a2, b2)
    print(f"{kind}:  angle {angle:.1f}°,  shortest distance {d:.3f}")
    if kind == "skew": print("skew lines have no meeting point")
    if kind == "they meet":
        P_, _ = closest(a1, b1, a2, b2); print("meeting point:", show(P_))
    if point.strip():
        try:
            pt = tuple(numbers(point)[:3]); assert len(pt) == 3
        except Exception:
            print("Type three numbers for your point."); return
        o1, o2 = on_line(pt, a1, b1), on_line(pt, a2, b2)
        if o1 and o2: print(f"✓ {show(pt)} is on both lines")
        else: print(f"✗ no: {show(pt)} is on line 1: {'yes' if o1 else 'no'}, on line 2: {'yes' if o2 else 'no'}")

def checker(line_1, line_2, point=""):
    """Three boxes and a check button; the verdict prints under them."""
    boxes = [Text(value=line_1, description="line 1", layout=Layout(width="330px")),
             Text(value=line_2, description="line 2", layout=Layout(width="330px")),
             Text(value=point, description="point", layout=Layout(width="240px"))]
    check_button, answer = Button(description="check"), Output()
    def on_check(_=None):
        answer.clear_output()
        with answer: check_your_answer(*(b_.value for b_ in boxes))
    check_button.on_click(on_check)
    display(VBox([HBox(boxes[:2]), HBox([boxes[2], check_button]), answer]))
    on_check()

In [ ]:
checker("1 0 -1 ; 1 1 2", "1 3 2 ; 2 -1 1", "3 2 3")

## 8 · Turn one line until they meet

The trap pair again. Drag the slider to turn the second line: its direction is (1, 2, c). The coral link is the
shortest gap between the lines. Watch it shrink to nothing as the lines meet.

In [ ]:
A1, B1, A2 = (1, -1, 2), (2, 3, 1), (2, 1, 0)      # the trap pair (this cell stands on its own)

def turn(c=-1.0, view=-55):
    a1, b1, a2, d2 = A1, B1, A2, (1, 2, c)
    P1, P2 = closest(a1, b1, a2, d2)
    n = cross(b1, d2)
    gap = abs(dot(sub(a2, a1), n)) / length(n)
    fig = plt.figure(figsize=(8.6, 5.6))
    ax = fig.add_subplot(projection="3d")
    space(ax, 18, view)
    ts = np.linspace(-1.6, 1.6, 2)
    ax.plot(*[[a1[i] + t * b1[i] for t in ts] for i in range(3)], color=CHALK, label="line 1")
    ax.plot(*[[a2[i] + (t - 1) * d2[i] for t in ts] for i in range(3)], color=TEAL, label=f"line 2: direction {show(d2)}")
    ax.plot(*zip(P1, P2), color=CORAL, lw=4)
    ax.scatter(*P1, color=CORAL, s=25); ax.scatter(*P2, color=CORAL, s=25)
    ax.set_xlim(-2, 4); ax.set_ylim(-5, 4); ax.set_zlim(-1, 5)
    ax.legend(loc="upper left", fontsize=11)
    ax.set_title("the lines meet" if gap < 1e-6 else f"shortest link: {gap:.3f}", color=CORAL)
    plt.show()

In [ ]:
interact(turn, c=FloatSlider(value=-1.0, min=-2.5, max=0.0, step=0.05, description="turn line 2",
                             layout=Layout(width="520px")),
         view=FloatSlider(value=-55, min=-180, max=180, step=10, description="view: turn"));

**Try it.** The challenge from the video: find the image of the point (1, 9, 2) in the line
(x − 1)/2 = y/1 = (z − 2)/(−2). Find the foot first (section 6 shows how); it is the midpoint.

## Sources
NCERT Mathematics Class XII, ch. 11 (Three Dimensional Geometry); CBSE Mathematics syllabus 2026–27 and sample
question paper 2025–26. All retrieved 2026-10-26. Every problem is our own.